# 05b - Content Representation: Model Selection (Round 2)

## Objective

We previously compared several sentence-embedding models and found `sentence-transformers/all-mpnet-base-v2` to be the most promising candidate for the current benchmark.

This notebook keeps MPNet as the reference model and evaluates additional sentence-embedding models to determine whether another representation provides a better trade-off for semantic content.

For a transcription $W$, we consider:

$$
Z_{\mathrm{content}}(W)=E(W).
$$

The benchmark is designed around five properties:

1. **Paraphrases** should have high cosine similarity.
2. **Contradictions / negations** should be substantially less similar than paraphrases.
3. **Related but different content** should show some semantic similarity.
4. **Unrelated sentences** should have low similarity.
5. **Lexically overlapping but semantically different sentences** should not be considered highly similar simply because they share many words.


This notebook only evaluates the text encoder. ASR is handled separately.


## Models

`sentence-transformers/all-mpnet-base-v2` is kept as the reference model from the first benchmark.

The additional candidates are:

- `sentence-transformers/nli-mpnet-base-v2`
- `sentence-transformers/nli-roberta-base-v2`
- `sentence-transformers/nli-distilroberta-base-v2`

These models provide different training objectives and architectures from the previous candidates. The goal is not to assume that a model is better from its general benchmark reputation, but to test the behavior that matters for this project.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer


/opt/anaconda3/envs/interview-anon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
MODELS = {
    "MPNet": "sentence-transformers/all-mpnet-base-v2",
    "NLI-MPNet": "sentence-transformers/nli-mpnet-base-v2",
    "NLI-RoBERTa": "sentence-transformers/nli-roberta-base-v2",
    "NLI-DistilRoBERTa": "sentence-transformers/nli-distilroberta-base-v2",
}

print("Models to evaluate:")
for name, model_name in MODELS.items():
    print(f"- {name}: {model_name}")

Models to evaluate:
- MPNet: sentence-transformers/all-mpnet-base-v2
- NLI-MPNet: sentence-transformers/nli-mpnet-base-v2
- NLI-RoBERTa: sentence-transformers/nli-roberta-base-v2
- NLI-DistilRoBERTa: sentence-transformers/nli-distilroberta-base-v2


## Controlled benchmark

The same benchmark structure is used for every model so that the comparison is fair.

In [3]:
benchmark = {
    "paraphrase": [
        (
            "I am studying statistics at EPFL.",
            "I study statistics at EPFL."
        ),
        (
            "I went to the supermarket yesterday.",
            "Yesterday, I bought some groceries."
        ),
        (
            "The meeting was postponed until Monday.",
            "The meeting has been moved to Monday."
        ),
    ],

    "contradiction": [
        (
            "I am studying statistics at EPFL.",
            "I do not study statistics at EPFL."
        ),
        (
            "The interview was recorded in Lausanne.",
            "The interview was not recorded in Lausanne."
        ),
        (
            "She works in finance.",
            "She does not work in finance."
        ),
        (
            "It rained the whole day.",
            "It did not rain during the day."
        ),
    ],

    "related": [
        (
            "The weather is very nice today.",
            "It rained the whole day."
        ),
        (
            "I am studying statistics at EPFL.",
            "I am taking a machine learning course at EPFL."
        ),
        (
            "I bought a new computer yesterday.",
            "My laptop stopped working this morning."
        ),
    ],

    "unrelated": [
        (
            "I am studying statistics at EPFL.",
            "The football match was cancelled because of the rain."
        ),
        (
            "My favorite programming language is Python.",
            "The restaurant closes at eleven tonight."
        ),
        (
            "I bought a new computer yesterday.",
            "The train arrived twenty minutes late."
        ),
    ],

    "lexical_overlap": [
        (
            "The bank is closed because it is Sunday.",
            "The bank is open because it is Sunday."
        ),
        (
            "I went to the bank to deposit some money.",
            "I sat on the bank of the river and watched the water."
        ),
        (
            "The crane lifted the heavy machine.",
            "The crane flew over the building."
        ),
        (
            "I saw a bat flying outside my house.",
            "He bought a new bat for the baseball game."
        ),
        (
            "The mouse is under the table.",
            "I bought a new mouse for my computer."
        ),
    ],
}

print("Number of pairs:")
for category, pairs in benchmark.items():
    print(f"{category}: {len(pairs)}")

Number of pairs:
paraphrase: 3
contradiction: 4
related: 3
unrelated: 3
lexical_overlap: 5


## Evaluation function

For each model, we encode both sentences in every pair and compute their cosine similarity.

Embeddings are L2-normalized so cosine similarity directly measures the angle between the resulting representation vectors.

In [4]:
def evaluate_model(model, benchmark):
    rows = []

    for category, pairs in benchmark.items():
        sentences = []
        for s1, s2 in pairs:
            sentences.extend([s1, s2])

        embeddings = model.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=False,
        )

        for i, (s1, s2) in enumerate(pairs):
            z1 = embeddings[2 * i]
            z2 = embeddings[2 * i + 1]
            similarity = float(np.dot(z1, z2))

            rows.append({
                "category": category,
                "sentence_1": s1,
                "sentence_2": s2,
                "cosine": similarity,
            })

    return pd.DataFrame(rows)

## Evaluate all models

The models are loaded and evaluated one at a time to avoid keeping all model weights in memory simultaneously.

In [5]:
all_results = []

for model_label, model_name in MODELS.items():
    print("=" * 80)
    print(f"Loading: {model_label}")
    print(f"Model:  {model_name}")

    model = SentenceTransformer(
    model_name,
    device="cpu",
    )

    embedding_dim = model.get_embedding_dimension()
    print(f"Embedding dimension: {embedding_dim}")

    results = evaluate_model(model, benchmark)
    results["model"] = model_label
    results["embedding_dim"] = embedding_dim
    all_results.append(results)

    del model

results_df = pd.concat(all_results, ignore_index=True)

print("\nFinished.")
print("Number of evaluated pairs:", len(results_df))

Loading: MPNet
Model:  sentence-transformers/all-mpnet-base-v2


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5705.60it/s]


Embedding dimension: 768
Loading: NLI-MPNet
Model:  sentence-transformers/nli-mpnet-base-v2


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5495.89it/s]


Embedding dimension: 768
Loading: NLI-RoBERTa
Model:  sentence-transformers/nli-roberta-base-v2


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 5968.47it/s]


Embedding dimension: 768
Loading: NLI-DistilRoBERTa
Model:  sentence-transformers/nli-distilroberta-base-v2


Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 5497.40it/s]


Embedding dimension: 768

Finished.
Number of evaluated pairs: 72


## Pair-level results

The complete pair-level results are retained so that we can inspect which examples drive the model differences.

In [6]:
display(
    results_df[
        ["model", "category", "cosine", "sentence_1", "sentence_2"]
    ].sort_values(["model", "category"])
)

,model,category,cosine,sentence_1,sentence_2
3,MPNet,contradiction,0.606829,I am studying statistics at EPFL.,I do not study statistics at EPFL.
4,MPNet,contradiction,0.646971,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
5,MPNet,contradiction,0.692017,She works in finance.,She does not work in finance.
6,MPNet,contradiction,0.479438,It rained the whole day.,It did not rain during the day.
13,MPNet,lexical_overlap,0.739567,The bank is closed because it is Sunday.,The bank is open because it is Sunday.
...,...,...,...,...,...
44,NLI-RoBERTa,related,0.711808,I am studying statistics at EPFL.,I am taking a machine learning course at EPFL.
45,NLI-RoBERTa,related,0.412790,I bought a new computer yesterday.,My laptop stopped working this morning.
46,NLI-RoBERTa,unrelated,0.098262,I am studying statistics at EPFL.,The football match was cancelled because of th...
47,NLI-RoBERTa,unrelated,0.006904,My favorite programming language is Python.,The restaurant closes at eleven tonight.


## Summary statistics

For every model and category we report the mean, standard deviation, minimum and maximum cosine similarity.

In [7]:
summary = (
    results_df
    .groupby(["model", "category"])["cosine"]
    .agg(["mean", "std", "min", "max"])
    .round(3)
)

display(summary)

mean    std    min    max
model             category                                   
MPNet             contradiction    0.606  0.091  0.479  0.692
                  lexical_overlap  0.518  0.142  0.405  0.740
                  paraphrase       0.787  0.080  0.709  0.869
                  related          0.418  0.185  0.212  0.573
                  unrelated        0.025  0.093 -0.072  0.114
NLI-DistilRoBERTa contradiction    0.536  0.052  0.467  0.590
                  lexical_overlap  0.465  0.239  0.187  0.730
                  paraphrase       0.863  0.092  0.796  0.968
                  related          0.456  0.249  0.298  0.743
                  unrelated        0.059  0.022  0.045  0.085
NLI-MPNet         contradiction    0.474  0.053  0.412  0.534
                  lexical_overlap  0.464  0.217  0.258  0.728
                  paraphrase       0.869  0.073  0.804  0.947
                  related          0.437  0.208  0.239  0.653
                  unrelated        0.010  0.029 -0.012  0.043
NLI-RoBERTa       contradiction    0.485  0.117  0.310  0.559
                  lexical_overlap  0.448  0.212  0.208  0.689
                  paraphrase       0.847  0.104  0.772  0.966
                  related          0.456  0.238  0.243  0.712
                  unrelated        0.036  0.054  0.002  0.098

## Diagnostic score

A single score is used only as a compact diagnostic, not as a definitive model-selection criterion.

We want:

- high similarity for paraphrases;
- a clear decrease for contradictions;
- a clear decrease for lexical-overlap cases;
- low similarity for unrelated pairs;
- related pairs to remain more similar than unrelated pairs.

We therefore compute:

$$
D = (P-C) + (P-L) + (R-U),
$$

where $P,C,L,R,U$ are the mean cosine similarities for paraphrase, contradiction, lexical overlap, related and unrelated pairs respectively.

A larger value indicates stronger separation under this particular benchmark. The individual category means should always be inspected alongside $D$.

In [8]:
means = (
    results_df
    .groupby(["model", "category"])["cosine"]
    .mean()
    .unstack()
)

diagnostic = pd.DataFrame(index=means.index)
diagnostic["paraphrase"] = means["paraphrase"]
diagnostic["contradiction"] = means["contradiction"]
diagnostic["lexical_overlap"] = means["lexical_overlap"]
diagnostic["related"] = means["related"]
diagnostic["unrelated"] = means["unrelated"]

diagnostic["P-C"] = diagnostic["paraphrase"] - diagnostic["contradiction"]
diagnostic["P-L"] = diagnostic["paraphrase"] - diagnostic["lexical_overlap"]
diagnostic["R-U"] = diagnostic["related"] - diagnostic["unrelated"]
diagnostic["diagnostic_score"] = (
    diagnostic["P-C"]
    + diagnostic["P-L"]
    + diagnostic["R-U"]
)

display(diagnostic.round(3).sort_values("diagnostic_score", ascending=False))

,paraphrase,contradiction,lexical_overlap,related,unrelated,P-C,P-L,R-U,diagnostic_score
model,,,,,,,,,
NLI-MPNet,0.869,0.474,0.464,0.437,0.010,0.395,0.405,0.427,1.226
NLI-RoBERTa,0.847,0.485,0.448,0.456,0.036,0.363,0.399,0.420,1.182
NLI-DistilRoBERTa,0.863,0.536,0.465,0.456,0.059,0.327,0.398,0.397,1.122
MPNet,0.787,0.606,0.518,0.418,0.025,0.181,0.270,0.392,0.843


## Direct comparison with MPNet

MPNet is the reference model from the first benchmark. The table below reports how each candidate changes the mean similarity in every category relative to MPNet.


In [9]:
mpnet_means = means.loc["MPNet"]

comparison = means.subtract(mpnet_means, axis="columns").round(3)
comparison = comparison.drop(index="MPNet")

display(comparison)

category,contradiction,lexical_overlap,paraphrase,related,unrelated
model,,,,,
NLI-DistilRoBERTa,-0.070,-0.053,0.076,0.039,0.034
NLI-MPNet,-0.133,-0.054,0.082,0.019,-0.015
NLI-RoBERTa,-0.122,-0.070,0.060,0.038,0.011


## Inspect the contradiction and lexical-overlap cases

These are the two categories that were particularly problematic for MiniLM and BGE in the first benchmark.

In [10]:
focus = results_df[
    results_df["category"].isin(["contradiction", "lexical_overlap"])
][["model", "category", "cosine", "sentence_1", "sentence_2"]]

display(focus.sort_values(["category", "model", "cosine"]))

,model,category,cosine,sentence_1,sentence_2
6,MPNet,contradiction,0.479438,It rained the whole day.,It did not rain during the day.
3,MPNet,contradiction,0.606829,I am studying statistics at EPFL.,I do not study statistics at EPFL.
4,MPNet,contradiction,0.646971,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
5,MPNet,contradiction,0.692017,She works in finance.,She does not work in finance.
60,NLI-DistilRoBERTa,contradiction,0.467302,It rained the whole day.,It did not rain during the day.
59,NLI-DistilRoBERTa,contradiction,0.531803,She works in finance.,She does not work in finance.
57,NLI-DistilRoBERTa,contradiction,0.557014,I am studying statistics at EPFL.,I do not study statistics at EPFL.
58,NLI-DistilRoBERTa,contradiction,0.589833,The interview was recorded in Lausanne.,The interview was not recorded in Lausanne.
24,NLI-MPNet,contradiction,0.412164,It rained the whole day.,It did not rain during the day.
23,NLI-MPNet,contradiction,0.450775,She works in finance.,She does not work in finance.


## Interpretation

- The benchmark is intentionally small and controlled. It is useful for screening candidate representations.

- cosine similarity is not expected to map directly to logical contradiction. A contradiction can still have a high similarity because the two sentences share topic and vocabulary.

-> The purpose of this benchmark is therefore to identify a representation that gives a useful separation between paraphrases, contradictions, lexical-overlap cases and unrelated content before moving to the audio-to-text pipeline.

The selected encoder will subsequently be used in the ASR pipeline:

$$
Z_{\mathrm{content}}(X)
=
E\left(\mathrm{ASR}(X)\right).
$$
